In [2]:
import pandas as pd
import numpy as np

## Why Extension Types Exist

pandas was originally built on NumPy, which led to several limitations:

- **Missing integers and Booleans** — NumPy has no null representation for these types, so pandas was forced to upcast to `float64` and use `np.nan`. This introduced subtle bugs in many algorithms.
- **String data** — stored as object arrays of Python strings, which is memory-heavy and slow.
- **Special types** (timezones, timedeltas) — required expensive Python object arrays.

pandas developed an **extension type system** that allows first-class data types not natively supported by NumPy. These types use a special `pd.NA` sentinel instead of `np.nan`.

### `pd.NA` vs `np.nan`

| | `np.nan` | `pd.NA` |
|--|----------|---------|
| Type | float | NA sentinel |
| Forces float upcast | Yes | No |
| Used by | Legacy NumPy-based dtypes | Extension dtypes |
| Display | `NaN` | `<NA>` |

### Specifying Extension Types

- Pass the type object: `dtype=pd.Int64Dtype()`
- Pass a string shorthand: `dtype="Int64"` (capitalization matters — lowercase `"int64"` is the NumPy type)

## Key Points

- Legacy pandas upcasts integer/boolean columns to `float64` when missing data is present.
- Extension types preserve the original dtype and use `pd.NA` for missing values.
- `pd.NA` is a dedicated NA sentinel — distinct from `np.nan`.
- Capitalized string shorthands (e.g. `"Int64"`, `"boolean"`) refer to extension types; lowercase (`"int64"`) refers to NumPy types.

In [3]:
# Legacy behavior: None forces upcast to float64
s = pd.Series([1, 2, 3, None])

print(s)
print(s.dtype)

0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64
float64


In [4]:
# Extension type: preserves Int64, uses <NA> instead of NaN
s = pd.Series([1, 2, 3, None], dtype=pd.Int64Dtype())

print(s)
print(s.dtype)

0       1
1       2
2       3
3    <NA>
dtype: Int64
Int64


In [5]:
s.isna()

0    False
1    False
2    False
3     True
dtype: bool

In [6]:
# The missing value is pd.NA, not np.nan
print(s[3])
print(s[3] is pd.NA)

<NA>
True


In [7]:
# String shorthand — equivalent to pd.Int64Dtype()
s = pd.Series([1, 2, 3, None], dtype="Int64")

s

0       1
1       2
2       3
3    <NA>
dtype: Int64

In [8]:
# String extension type — more memory-efficient than object arrays
s = pd.Series(["one", "two", None, "three"], dtype=pd.StringDtype())

print(s)
print(s.dtype)

0      one
1      two
2     <NA>
3    three
dtype: string
string


## Converting to Extension Types with `astype`

Extension types can be applied to existing columns using `astype`, making them easy to integrate into a data cleaning pipeline.

```python
df["A"] = df["A"].astype("Int64")     # nullable integer
df["B"] = df["B"].astype("string")    # efficient string type
df["C"] = df["C"].astype("boolean")   # nullable boolean
```

After conversion:
- Columns retain their logical type even when missing values are present.
- Missing values are displayed as `<NA>` rather than `NaN` or `None`.

### Extension Type Reference

| Extension Type | String Shorthand | Description |
|----------------|-----------------|-------------|
| `BooleanDtype` | `"boolean"` | Nullable Boolean |
| `CategoricalDtype` | `"category"` | Categorical data |
| `DatetimeTZDtype` | — | Datetime with time zone |
| `StringDtype` | `"string"` | Efficient nullable string (requires `pyarrow`) |
| `Float32Dtype` | `"Float32"` | 32-bit nullable float |
| `Float64Dtype` | `"Float64"` | 64-bit nullable float |
| `Int8Dtype` | `"Int8"` | 8-bit nullable signed integer |
| `Int16Dtype` | `"Int16"` | 16-bit nullable signed integer |
| `Int32Dtype` | `"Int32"` | 32-bit nullable signed integer |
| `Int64Dtype` | `"Int64"` | 64-bit nullable signed integer |
| `UInt8Dtype` | `"UInt8"` | 8-bit nullable unsigned integer |
| `UInt16Dtype` | `"UInt16"` | 16-bit nullable unsigned integer |
| `UInt32Dtype` | `"UInt32"` | 32-bit nullable unsigned integer |
| `UInt64Dtype` | `"UInt64"` | 64-bit nullable unsigned integer |

## Key Points

- `astype("Int64")` / `astype("boolean")` / `astype("string")` convert columns to extension types.
- After conversion, missing values appear as `<NA>` regardless of the original sentinel.
- Signed integer types use capital `I` (`"Int64"`); unsigned use capital `U` (`"UInt64"`).
- `"string"` type requires `pyarrow` and is more efficient than object-dtype string columns.

In [9]:
df = pd.DataFrame({
    "A": [1, 2, None, 4],
    "B": ["one", "two", "three", None],
    "C": [False, None, False, True]
})

df

,A,B,C
0,1.0,one,False
1,2.0,two,None
2,NaN,three,False
3,4.0,None,True


In [10]:
df["A"] = df["A"].astype("Int64")
df["B"] = df["B"].astype("string")
df["C"] = df["C"].astype("boolean")

df

,A,B,C
0,1,one,False
1,2,two,<NA>
2,<NA>,three,False
3,4,<NA>,True


In [11]:
df.dtypes

A             Int64
B    string[python]
C           boolean
dtype: object